In [25]:
#########################################################################
###   The Jupyter notebook calculates the final dataset from the 
###   TADPOLE_D1_D2.csv file and generates the ADNI_3C.csv
###   
###   Federal Institute of Education, Science, and Technology 
###   RG Saraiva Jr
###   2026
#########################################################################

In [26]:
import pandas as pd

In [27]:
# Read the ADNI file TADPOLE_D1_D2
df_temp = pd.read_csv("TADPOLE_D1_D2.csv",low_memory=False)

In [28]:
# Select the features of interest
df_trab = df_temp[['RID','EXAMDATE','Ventricles','Hippocampus','WholeBrain',
                   'Entorhinal','Fusiform','MidTemp','ICV','CDRSB','MMSE','RAVLT_learning',
                   'RAVLT_immediate','RAVLT_forgetting','RAVLT_perc_forgetting_bl',
                   'ADAS11','ADAS13','FAQ','FDG','AV45','APOE4','AGE','PTEDUCAT','DX']]

In [29]:
# Removes records with missing data (NAN)
dataset = df_trab.dropna(axis=0)

In [30]:
# Reset the index
dataset = dataset.reset_index(drop=True)

In [31]:
# Return only one instance per patient—retrieve the patient's most recent appointment from the database
vc = dataset['RID'].value_counts()
df_final = pd.DataFrame(columns=dataset.columns)
for i in range(len(vc)):
    df_paciente = dataset[dataset['RID'] == vc.index[i]]
    df_linha = df_paciente[df_paciente['EXAMDATE'] == dataset[dataset['RID'] == vc.index[i]]['EXAMDATE'].max()]
    df_final = pd.concat([df_final, df_linha],ignore_index=True)    
dataset = df_final

C:\Users\j_df\AppData\Local\Temp\ipykernel_22484\709234824.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final, df_linha],ignore_index=True)


In [32]:
# Exclude patients with a transition from DEMENTIA to MCI
df_remove = dataset.loc[dataset['DX'] == 'Dementia to MCI']
dataset = dataset.drop(df_remove.index)

In [33]:
# Excludes patients who have progressed from MCI to dementia
df_remove = dataset.loc[dataset['DX'] == 'MCI to NL']
dataset = dataset.drop(df_remove.index)
df_final = dataset

In [34]:
# Group and number the CN, MCI, and AD classes
# CN  = 0  (NL)
# MCI = 1  (MCI)
# AD  = 2  (Dementia)
for i in range(0,len(df_final)):
    if df_final.iloc[i,-1] == 'NL':
        df_final.iloc[i,-1] = 0
    if df_final.iloc[i,-1] == 'MCI':
        df_final.iloc[i,-1] = 1
    if df_final.iloc[i,-1] == 'NL to MCI':
        df_final.iloc[i,-1] = 1
    if df_final.iloc[i,-1] == 'Dementia':
        df_final.iloc[i,-1] = 2    
    if df_final.iloc[i,-1] == 'MCI to Dementia':
        df_final.iloc[i,-1] = 2  

In [35]:
# Remove the patient ID and the exam date
df_final = df_final.drop(['RID','EXAMDATE'], axis=1)

In [36]:
# Save the resulting file as a CSV
df_final.to_csv('ADNI_3C.csv',index = False)

In [37]:
# Calculates the number of instances per class
df_final["DX"].value_counts()

DX
1    396
0    291
2    148
Name: count, dtype: int64